In [ ]:
# LOAD DATA + PREVIOUS RESULTS

import os
import pandas as pd
import numpy as np

RESULT_DIR = "/kaggle/working/results"
os.makedirs(RESULT_DIR, exist_ok=True)

DATA_DIR = "/kaggle/input/notebooks/nguynthominhqunhm15/nlp-tv1/data"
PREV_RESULT_DIR = "/kaggle/input/notebooks/nguynthominhqunhm15/nlp-tv1/results"

train_prepared = pd.read_csv(f"{DATA_DIR}/train_prepared.csv")
val_prepared = pd.read_csv(f"{DATA_DIR}/val_prepared.csv")
test_prepared = pd.read_csv(f"{DATA_DIR}/test_prepared.csv")

val_comp = pd.read_csv(
    f"{PREV_RESULT_DIR}/baseline_vs_final_phobert_validation_comparison.csv"
)

test_comp = pd.read_csv(
    f"{PREV_RESULT_DIR}/baseline_vs_final_phobert_test_comparison.csv"
)

aspect_list = [
    "SCREEN", "CAMERA", "FEATURES", "BATTERY",
    "PERFORMANCE", "STORAGE", "DESIGN",
    "PRICE", "GENERAL", "SER&ACC"
]

label_ids = [0, 1, 2, 3]
label_names = ["Negative", "Neutral", "Positive", "not_mentioned"]

baseline_val_eval_df = val_comp[[
    "aspect",
    "accuracy_baseline_val",
    "macro_precision_baseline_val",
    "macro_recall_baseline_val",
    "macro_f1_baseline_val"
]].rename(columns={
    "accuracy_baseline_val": "accuracy",
    "macro_precision_baseline_val": "macro_precision",
    "macro_recall_baseline_val": "macro_recall",
    "macro_f1_baseline_val": "macro_f1"
})

phobert_val_df = val_comp[[
    "aspect",
    "accuracy_phobert_val",
    "macro_precision_phobert_val",
    "macro_recall_phobert_val",
    "macro_f1_phobert_val"
]].rename(columns={
    "accuracy_phobert_val": "accuracy",
    "macro_precision_phobert_val": "macro_precision",
    "macro_recall_phobert_val": "macro_recall",
    "macro_f1_phobert_val": "macro_f1"
})

baseline_test_eval_df = test_comp[[
    "aspect",
    "accuracy_baseline_test",
    "macro_precision_baseline_test",
    "macro_recall_baseline_test",
    "macro_f1_baseline_test"
]].rename(columns={
    "accuracy_baseline_test": "accuracy",
    "macro_precision_baseline_test": "macro_precision",
    "macro_recall_baseline_test": "macro_recall",
    "macro_f1_baseline_test": "macro_f1"
})

phobert_test_df = test_comp[[
    "aspect",
    "accuracy_phobert_test",
    "macro_precision_phobert_test",
    "macro_recall_phobert_test",
    "macro_f1_phobert_test"
]].rename(columns={
    "accuracy_phobert_test": "accuracy",
    "macro_precision_phobert_test": "macro_precision",
    "macro_recall_phobert_test": "macro_recall",
    "macro_f1_phobert_test": "macro_f1"
})

print(train_prepared.shape, val_prepared.shape, test_prepared.shape)
print("Loaded previous results.")

(7784, 11) (1669, 11) (1669, 11)
Loaded previous results.


In [ ]:

# 8. MULTI-TASK + HYBRID PHOBERT - FIXED VERSION


import os
import gc
import random
import numpy as np
import pandas as pd
import torch
from torch import nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "vinai/phobert-base"


MTL_DIR = f"{RESULT_DIR}/phobert_multitask"
MTL_ENCODER_DIR = f"{RESULT_DIR}/phobert_mtl_encoder"
HYBRID_MODEL_DIR = f"{RESULT_DIR}/phobert_hybrid_models"

os.makedirs(MTL_DIR, exist_ok=True)
os.makedirs(MTL_ENCODER_DIR, exist_ok=True)
os.makedirs(HYBRID_MODEL_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# UTILS

def get_class_weights(train_df, aspect, max_weight=10):
    y = train_df[f"{aspect}_id"].values
    classes = np.array(label_ids)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y
    )

    weights = np.clip(weights, 1.0, max_weight)

    return torch.tensor(weights, dtype=torch.float)


def compute_single_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    p, r, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1
    }


def compute_single_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    p, r, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    return acc, p, r, f1

In [ ]:
# SMART OVERSAMPLING FOR WEAK ASPECTS

def smart_oversample_aspect(
    df,
    aspect,
    min_samples=800,
    max_total_multiplier=2.0,
    random_state=42
):
    """
    Oversample những class nhỏ của 1 aspect.
    Vẫn giữ toàn bộ các label aspect khác, nên dùng được cho multi-task.
    """
    col = f"{aspect}_id"
    df_base = df.copy()

    original_len = len(df_base)
    max_len = int(original_len * max_total_multiplier)

    parts = [df_base]

    counts = df_base[col].value_counts().to_dict()

    for label in label_ids:
        count = counts.get(label, 0)

        if count == 0:
            continue

        if count < min_samples:
            need = min_samples - count

            df_label = df_base[df_base[col] == label]

            sampled = df_label.sample(
                n=need,
                replace=True,
                random_state=random_state
            )

            parts.append(sampled)

    df_out = pd.concat(parts, axis=0)

    if len(df_out) > max_len:
        df_out = df_out.sample(
            n=max_len,
            replace=False,
            random_state=random_state
        )

    df_out = df_out.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    return df_out


def build_multitask_train_df(train_df):
    """
    Boost nhẹ các aspect yếu nhưng vẫn train full 10 aspect.
    """
    df_mt = train_df.copy()

    df_mt = smart_oversample_aspect(
        df_mt,
        aspect="STORAGE",
        min_samples=600,
        max_total_multiplier=1.7,
        random_state=SEED
    )

    df_mt = smart_oversample_aspect(
        df_mt,
        aspect="PERFORMANCE",
        min_samples=900,
        max_total_multiplier=1.4,
        random_state=SEED
    )

    df_mt = smart_oversample_aspect(
        df_mt,
        aspect="GENERAL",
        min_samples=900,
        max_total_multiplier=1.4,
        random_state=SEED
    )

    return df_mt.reset_index(drop=True)

In [ ]:
# FOCAL LOSS
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.CrossEntropyLoss(
            weight=self.alpha,
            reduction="none"
        )(logits, targets)

        pt = torch.exp(-ce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        return focal_loss.mean()

In [ ]:
# MULTI-TASK DATASET
# labels shape: [batch, num_aspects]

def prepare_multitask_dataset(df):
    label_matrix = df[[f"{a}_id" for a in aspect_list]].values.astype(int)

    dataset = Dataset.from_dict({
        "text_clean": df["text_clean"].tolist(),
        "labels": label_matrix.tolist()
    })

    def tokenize_batch(examples):
        enc = tokenizer(
            examples["text_clean"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

        enc["labels"] = examples["labels"]

        return enc

    dataset = dataset.map(tokenize_batch, batched=True)

    dataset = dataset.remove_columns(["text_clean"])

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    return dataset

In [ ]:
# MULTI-TASK MODEL
# Shared PhoBERT encoder + 10 heads

class MultiTaskPhoBERT(nn.Module):
    def __init__(self, model_name, aspects, num_classes=4):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.aspects = aspects

        self.heads = nn.ModuleDict({
            aspect: nn.Linear(hidden_size, num_classes)
            for aspect in aspects
        })

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0]

        logits_list = []

        for aspect in self.aspects:
            logits = self.heads[aspect](cls_output)
            logits_list.append(logits)

        logits = torch.stack(logits_list, dim=1)

        return {
            "logits": logits
        }

In [ ]:
# MULTI-TASK METRICS
# output logits shape: [batch, 10, 4]
# labels shape: [batch, 10]

def compute_multitask_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=2)

    results = {}
    f1_list = []

    for i, aspect in enumerate(aspect_list):
        y_true = labels[:, i]
        y_pred = preds[:, i]

        f1 = f1_score(
            y_true,
            y_pred,
            labels=label_ids,
            average="macro",
            zero_division=0
        )

        acc = accuracy_score(y_true, y_pred)

        results[f"{aspect}_accuracy"] = acc
        results[f"{aspect}_macro_f1"] = f1

        f1_list.append(f1)

    results["macro_f1"] = float(np.mean(f1_list))

    return results

In [ ]:
#tùy chỉnh loss để tính riêng cho từng khía cạnh, kết hợp class weights và aspect weights.
class MultiTaskTrainer(Trainer):
    def __init__(self, aspect_weights, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.aspect_weights = aspect_weights
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits_all = outputs["logits"]

        total_loss = 0.0

        for i, aspect in enumerate(aspect_list):
            logits = logits_all[:, i, :]
            y = labels[:, i]

            alpha = self.class_weights[aspect].to(logits.device)

            loss_fct = FocalLoss(
                alpha=alpha,
                gamma=2.0
            )

            loss_i = loss_fct(logits, y)

            total_loss = total_loss + self.aspect_weights[aspect] * loss_i

        total_loss = total_loss / len(aspect_list)

        return (total_loss, outputs) if return_outputs else total_loss

In [10]:
aspect_weights = {
    "SCREEN": 1.0,
    "CAMERA": 1.0,
    "FEATURES": 1.0,
    "BATTERY": 1.0,
    "PERFORMANCE": 1.3,
    "STORAGE": 2.0,
    "DESIGN": 1.0,
    "PRICE": 1.0,
    "GENERAL": 1.3,
    "SER&ACC": 1.0
}

In [ ]:
# TRAIN MULTI-TASK PHOBERT

train_mt_df = build_multitask_train_df(train_prepared)

print("Original train size:", len(train_prepared))
print("Multi-task train size:", len(train_mt_df))

for aspect in ["STORAGE", "PERFORMANCE", "GENERAL"]:
    print("\nAspect:", aspect)
    print(train_mt_df[f"{aspect}_id"].value_counts().sort_index())

multitask_class_weights = {
    aspect: get_class_weights(
        train_mt_df,
        aspect,
        max_weight=10
    )
    for aspect in aspect_list
}

train_ds_mt = prepare_multitask_dataset(train_mt_df)
val_ds_mt = prepare_multitask_dataset(val_prepared)

model_mt = MultiTaskPhoBERT(
    MODEL_NAME,
    aspects=aspect_list,
    num_classes=4
)

training_args_mt = TrainingArguments(
    output_dir=MTL_DIR,

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    save_total_limit=1,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,

    report_to="none",
    seed=SEED
)

trainer_mt = MultiTaskTrainer(
    model=model_mt,
    args=training_args_mt,
    train_dataset=train_ds_mt,
    eval_dataset=val_ds_mt,
    compute_metrics=compute_multitask_metrics,
    aspect_weights=aspect_weights,
    class_weights=multitask_class_weights,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

trainer_mt.train()

mt_val_result = trainer_mt.evaluate()
mt_val_result

Original train size: 7784
Multi-task train size: 10408

Aspect: STORAGE
STORAGE_id
0     730
1     643
2     643
3    8392
Name: count, dtype: int64

Aspect: PERFORMANCE
PERFORMANCE_id
0    1817
1     945
2    2886
3    4760
Name: count, dtype: int64

Aspect: GENERAL
GENERAL_id
0    1145
1     900
2    4698
3    3665
Name: count, dtype: int64


Map:   0%|          | 0/10408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1669 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Screen Accuracy,Screen Macro F1,Camera Accuracy,Camera Macro F1,Features Accuracy,Features Macro F1,Battery Accuracy,Battery Macro F1,Performance Accuracy,Performance Macro F1,Storage Accuracy,Storage Macro F1,Design Accuracy,Design Macro F1,Price Accuracy,Price Macro F1,General Accuracy,General Macro F1,Ser&acc Accuracy,Ser&acc Macro F1,Macro F1
1,0.885623,0.654210,0.846016,0.486232,0.880767,0.680876,0.681246,0.514362,0.772319,0.650588,0.646495,0.575237,0.983823,0.483826,0.816657,0.451379,0.844218,0.684536,0.645896,0.553571,0.756741,0.469978,0.555059
2,0.508206,0.545482,0.910725,0.559719,0.896345,0.718575,0.817256,0.610790,0.840623,0.710722,0.677052,0.610804,0.982624,0.499575,0.907729,0.600753,0.877172,0.744666,0.690833,0.602761,0.838226,0.570263,0.622863
3,0.374555,0.479937,0.886159,0.583957,0.899341,0.729083,0.819053,0.634641,0.823847,0.716276,0.738766,0.664631,0.983823,0.486281,0.868185,0.619792,0.891552,0.769187,0.756141,0.661765,0.872978,0.595048,0.646066
4,0.286762,0.451996,0.902936,0.627820,0.904134,0.741025,0.839425,0.672502,0.871780,0.763069,0.763930,0.681400,0.983823,0.485491,0.909527,0.666188,0.907130,0.799330,0.768724,0.667848,0.863391,0.594060,0.669873
5,0.226298,0.452133,0.906531,0.637885,0.924506,0.790287,0.849011,0.683379,0.874176,0.771611,0.798083,0.719480,0.986818,0.464461,0.916117,0.683727,0.904733,0.791964,0.793289,0.692698,0.861594,0.588996,0.682449
6,0.183629,0.446659,0.931696,0.669354,0.927501,0.783571,0.852606,0.688282,0.893948,0.792271,0.813062,0.722602,0.988017,0.534543,0.918514,0.671662,0.905333,0.789902,0.791492,0.691525,0.882564,0.607805,0.695152
7,0.154215,0.442889,0.921510,0.654094,0.929898,0.789070,0.859197,0.699718,0.885560,0.786409,0.809467,0.720087,0.988017,0.534342,0.927501,0.683152,0.908328,0.797296,0.794488,0.701480,0.866387,0.605147,0.697079
8,0.136743,0.442894,0.930497,0.667607,0.927501,0.787276,0.857400,0.687760,0.887957,0.783594,0.801678,0.722844,0.989215,0.547093,0.926303,0.688003,0.908928,0.799315,0.798083,0.697704,0.881366,0.620264,0.700146


{'eval_loss': 0.44289442896842957,
 'eval_SCREEN_accuracy': 0.9304973037747154,
 'eval_SCREEN_macro_f1': 0.6676071420928188,
 'eval_CAMERA_accuracy': 0.9275014979029359,
 'eval_CAMERA_macro_f1': 0.7872758627597337,
 'eval_FEATURES_accuracy': 0.8573996405032954,
 'eval_FEATURES_macro_f1': 0.6877595574985393,
 'eval_BATTERY_accuracy': 0.8879568603954464,
 'eval_BATTERY_macro_f1': 0.7835938692804767,
 'eval_PERFORMANCE_accuracy': 0.8016776512881966,
 'eval_PERFORMANCE_macro_f1': 0.7228440141450622,
 'eval_STORAGE_accuracy': 0.9892150988615938,
 'eval_STORAGE_macro_f1': 0.5470926756096417,
 'eval_DESIGN_accuracy': 0.926303175554224,
 'eval_DESIGN_macro_f1': 0.6880028408919745,
 'eval_PRICE_accuracy': 0.9089275014979029,
 'eval_PRICE_macro_f1': 0.799314703733766,
 'eval_GENERAL_accuracy': 0.7980826842420611,
 'eval_GENERAL_macro_f1': 0.6977040395763064,
 'eval_SER&ACC_accuracy': 0.8813660874775314,
 'eval_SER&ACC_macro_f1': 0.6202637441476151,
 'eval_macro_f1': 0.7001458449735934,
 'eval_ru

In [ ]:
# SAVE MULTI-TASK ENCODER FOR HYBRID FINE-TUNING

trainer_mt.model.encoder.save_pretrained(MTL_ENCODER_DIR)
tokenizer.save_pretrained(MTL_ENCODER_DIR)

print("Saved MTL encoder:", MTL_ENCODER_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved MTL encoder: /kaggle/working/results/phobert_mtl_encoder


In [ ]:
# EVALUATE MULTI-TASK ON TEST

test_ds_mt = prepare_multitask_dataset(test_prepared)

mt_preds = trainer_mt.predict(test_ds_mt)

mt_logits = mt_preds.predictions
mt_labels = mt_preds.label_ids

mt_y_pred = np.argmax(mt_logits, axis=2)

multi_results = []

for i, aspect in enumerate(aspect_list):
    y_true = mt_labels[:, i]
    y_pred = mt_y_pred[:, i]

    acc = accuracy_score(y_true, y_pred)

    p, r, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    multi_results.append({
        "aspect": aspect,
        "accuracy": acc,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1
    })

multi_df = pd.DataFrame(multi_results)
multi_df.sort_values("macro_f1", ascending=False)

Map:   0%|          | 0/1669 [00:00<?, ? examples/s]

,aspect,accuracy,macro_precision,macro_recall,macro_f1
7,PRICE,0.913122,0.784086,0.857999,0.811931
1,CAMERA,0.932295,0.786520,0.833435,0.807404
3,BATTERY,0.899940,0.783547,0.824270,0.798544
0,SCREEN,0.935890,0.705442,0.770715,0.733489
2,FEATURES,0.872978,0.722921,0.755121,0.730868
4,PERFORMANCE,0.814859,0.718124,0.739569,0.727104
8,GENERAL,0.793289,0.708583,0.756586,0.725948
6,DESIGN,0.919113,0.669682,0.694178,0.680073
9,SER&ACC,0.876573,0.632627,0.696612,0.652131
5,STORAGE,0.991612,0.512044,0.727804,0.587631


In [ ]:
# HYBRID SINGLE-TASK FINE-TUNING FROM MTL ENCODER

def smart_oversample_single(
    df,
    aspect,
    min_samples=800,
    max_total_multiplier=2.0,
    random_state=42
):
    col = f"{aspect}_id"

    df_base = df.copy()
    original_len = len(df_base)
    max_len = int(original_len * max_total_multiplier)

    parts = [df_base]

    counts = df_base[col].value_counts().to_dict()

    for label in label_ids:
        count = counts.get(label, 0)

        if count == 0:
            continue

        if count < min_samples:
            need = min_samples - count

            df_label = df_base[df_base[col] == label]

            sampled = df_label.sample(
                n=need,
                replace=True,
                random_state=random_state
            )

            parts.append(sampled)

    df_out = pd.concat(parts, axis=0)

    if len(df_out) > max_len:
        df_out = df_out.sample(
            n=max_len,
            replace=False,
            random_state=random_state
        )

    df_out = df_out.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    return df_out


def prepare_single_dataset(df, aspect):
    dataset = Dataset.from_pandas(
        df[["text_clean", f"{aspect}_id"]]
        .rename(columns={f"{aspect}_id": "label"})
    )

    def tokenize_batch(examples):
        return tokenizer(
            examples["text_clean"],
            padding="max_length",
            truncation=True,
            max_length=128
        )

    dataset = dataset.map(tokenize_batch, batched=True)
    dataset = dataset.remove_columns(["text_clean"])

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )

    return dataset


class HybridTrainer(Trainer):
    def __init__(self, class_weights=None, loss_type="ce", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.loss_type = loss_type

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.loss_type == "focal":
            loss_fct = FocalLoss(
                alpha=self.class_weights.to(logits.device),
                gamma=2.0
            )
        else:
            loss_fct = nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device)
            )

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
# TRAIN HYBRID MODEL FOR EACH ASPECT

def train_hybrid_single_aspect(train_df, val_df, aspect):
    print("=" * 80)
    print(f"Hybrid fine-tuning from MTL encoder - {aspect}")

    if aspect == "STORAGE":
        train_used = smart_oversample_single(
            train_df,
            aspect,
            min_samples=1000,
            max_total_multiplier=2.0,
            random_state=SEED
        )
        loss_type = "focal"
        epochs = 5
        lr = 2e-5
        max_weight = 10

    elif aspect in ["PERFORMANCE", "GENERAL"]:
        train_used = smart_oversample_single(
            train_df,
            aspect,
            min_samples=800,
            max_total_multiplier=1.5,
            random_state=SEED
        )
        loss_type = "focal"
        epochs = 4
        lr = 2e-5
        max_weight = 10

    else:
        train_used = train_df.copy()
        loss_type = "ce"
        epochs = 3
        lr = 2e-5
        max_weight = 10

    print("\nTrain distribution:")
    print(train_used[f"{aspect}_id"].value_counts().sort_index())

    class_weights = get_class_weights(
        train_used,
        aspect,
        max_weight=max_weight
    )

    train_ds = prepare_single_dataset(train_used, aspect)
    val_ds = prepare_single_dataset(val_df, aspect)

    model = AutoModelForSequenceClassification.from_pretrained(
        MTL_ENCODER_DIR,
        num_labels=4
    )

    # output_dir = f"{HYBRID_MODEL_DIR}/tmp_{aspect}"

    import shutil

    output_dir = f"/tmp/hybrid_tmp_{aspect}"
    shutil.rmtree(output_dir, ignore_errors=True)

    training_args = TrainingArguments(
        output_dir=output_dir,

        do_train=True,
        do_eval=True,

        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_f1",
        greater_is_better=True,
        save_total_limit=1,

        learning_rate=lr,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=epochs,

        report_to="none",
        seed=SEED
    )

    trainer = HybridTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_single_metrics,
        class_weights=class_weights,
        loss_type=loss_type,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2
            )
        ]
    )

    trainer.train()
    val_result = trainer.evaluate()

    save_path = f"{HYBRID_MODEL_DIR}/phobert_hybrid_{aspect}"
    os.makedirs(save_path, exist_ok=True)

    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    print("Saved hybrid model:", save_path)
    print("Validation result:", val_result)

    return trainer, val_result, save_path

In [16]:
hybrid_val_results = []
hybrid_model_paths = {}

hybrid_aspects = [
    "SCREEN",
    "CAMERA",
    "FEATURES",
    "STORAGE",
    "PERFORMANCE",
    "GENERAL"
]

for aspect in hybrid_aspects:
    clear_gpu()

    trainer, val_result, save_path = train_hybrid_single_aspect(
        train_prepared,
        val_prepared,
        aspect
    )

    hybrid_model_paths[aspect] = save_path

    hybrid_val_results.append({
        "aspect": aspect,
        "accuracy": val_result["eval_accuracy"],
        "macro_precision": val_result["eval_macro_precision"],
        "macro_recall": val_result["eval_macro_recall"],
        "macro_f1": val_result["eval_macro_f1"],
        "model_path": save_path
    })

    del trainer
    clear_gpu()

hybrid_val_df = pd.DataFrame(hybrid_val_results)
hybrid_val_df.sort_values("macro_f1", ascending=False)

Hybrid fine-tuning from MTL encoder - SCREEN

Train distribution:
SCREEN_id
0     367
1      59
2     515
3    6843
Name: count, dtype: int64


Map:   0%|          | 0/7784 [00:00<?, ? examples/s]

Map:   0%|          | 0/1669 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: /kaggle/working/results/phobert_mtl_encoder
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved hybrid model: /kaggle/working/results/phobert_hybrid_models/phobert_hybrid_GENERAL
Validation result: {'eval_loss': 0.561619222164154, 'eval_accuracy': 0.8448172558418214, 'eval_macro_precision': 0.7511213550600343, 'eval_macro_recall': 0.7310460182013357, 'eval_macro_f1': 0.7371651653404903, 'eval_runtime': 10.2158, 'eval_samples_per_second': 163.374, 'eval_steps_per_second': 10.278, 'epoch': 4.0}


,aspect,accuracy,macro_precision,macro_recall,macro_f1,model_path
1,CAMERA,0.947274,0.829066,0.850622,0.839110,/kaggle/working/results/phobert_hybrid_models/...
4,PERFORMANCE,0.854404,0.772838,0.779651,0.776099,/kaggle/working/results/phobert_hybrid_models/...
2,FEATURES,0.893948,0.770332,0.767613,0.768525,/kaggle/working/results/phobert_hybrid_models/...
0,SCREEN,0.952666,0.724944,0.751149,0.737509,/kaggle/working/results/phobert_hybrid_models/...
5,GENERAL,0.844817,0.751121,0.731046,0.737165,/kaggle/working/results/phobert_hybrid_models/...
3,STORAGE,0.989814,0.498632,0.485501,0.484841,/kaggle/working/results/phobert_hybrid_models/...


In [17]:
extra_hybrid_aspects = [
    "BATTERY",
    "PRICE",
    "DESIGN",
    "SER&ACC"
]

for aspect in extra_hybrid_aspects:
    clear_gpu()

    trainer, val_result, save_path = train_hybrid_single_aspect(
        train_prepared,
        val_prepared,
        aspect
    )

    hybrid_model_paths[aspect] = save_path

    hybrid_val_results.append({
        "aspect": aspect,
        "accuracy": val_result["eval_accuracy"],
        "macro_precision": val_result["eval_macro_precision"],
        "macro_recall": val_result["eval_macro_recall"],
        "macro_f1": val_result["eval_macro_f1"],
        "model_path": save_path
    })

    del trainer
    clear_gpu()

Hybrid fine-tuning from MTL encoder - BATTERY

Train distribution:
BATTERY_id
0    1235
1     339
2    2016
3    4194
Name: count, dtype: int64


Map:   0%|          | 0/7784 [00:00<?, ? examples/s]

Map:   0%|          | 0/1669 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: /kaggle/working/results/phobert_mtl_encoder
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved hybrid model: /kaggle/working/results/phobert_hybrid_models/phobert_hybrid_SER&ACC
Validation result: {'eval_loss': 0.8413470387458801, 'eval_accuracy': 0.9029358897543439, 'eval_macro_precision': 0.6262636825734095, 'eval_macro_recall': 0.6371978770676401, 'eval_macro_f1': 0.6289572150607157, 'eval_runtime': 10.3738, 'eval_samples_per_second': 160.886, 'eval_steps_per_second': 10.122, 'epoch': 3.0}


In [ ]:

# EVALUATE HYBRID MODELS ON TEST


def evaluate_single_model(model_path, test_df, aspect):
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    trainer = Trainer(
        model=model,
        compute_metrics=compute_single_metrics
    )

    test_ds = prepare_single_dataset(test_df, aspect)

    preds = trainer.predict(test_ds)

    y_pred = np.argmax(preds.predictions, axis=1)
    y_true = preds.label_ids

    acc, p, r, f1 = compute_single_metrics_from_pred(
        y_true,
        y_pred
    )

    del trainer
    del model
    clear_gpu()

    return y_true, y_pred, acc, p, r, f1


hybrid_test_results = []

#  dùng toàn bộ model đã train
for aspect in hybrid_model_paths.keys():

    print("=" * 80)
    print(f"Hybrid Test Evaluation - {aspect}")

    model_path = hybrid_model_paths[aspect]

    y_true, y_pred, acc, p, r, f1 = evaluate_single_model(
        model_path,
        test_prepared,
        aspect
    )

    hybrid_test_results.append({
        "aspect": aspect,
        "accuracy": acc,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
        "model_path": model_path
    })

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=label_names,
        zero_division=0
    ))

hybrid_test_df = pd.DataFrame(hybrid_test_results)

hybrid_test_df = hybrid_test_df.sort_values(
    "macro_f1",
    ascending=False
).reset_index(drop=True)

hybrid_test_df

Hybrid Test Evaluation - SCREEN


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/1669 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.9497
Macro F1: 0.7388

Classification Report:
               precision    recall  f1-score   support

     Negative       0.72      0.74      0.73        85
      Neutral       0.46      0.40      0.43        15
     Positive       0.78      0.86      0.82        97
not_mentioned       0.98      0.97      0.98      1472

     accuracy                           0.95      1669
    macro avg       0.74      0.74      0.74      1669
 weighted avg       0.95      0.95      0.95      1669

Hybrid Test Evaluation - CAMERA


...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/1669 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.8975
Macro F1: 0.6723

Classification Report:
               precision    recall  f1-score   support

     Negative       0.68      0.50      0.58       129
      Neutral       0.21      0.39      0.27        18
     Positive       0.89      0.90      0.90       304
not_mentioned       0.93      0.95      0.94      1218

     accuracy                           0.90      1669
    macro avg       0.68      0.68      0.67      1669
 weighted avg       0.90      0.90      0.90      1669



,aspect,accuracy,macro_precision,macro_recall,macro_f1,model_path
0,CAMERA,0.946075,0.835552,0.860698,0.847405,/kaggle/working/results/phobert_hybrid_models/...
1,BATTERY,0.925704,0.832091,0.849470,0.840429,/kaggle/working/results/phobert_hybrid_models/...
2,PRICE,0.916117,0.790174,0.853382,0.811317,/kaggle/working/results/phobert_hybrid_models/...
3,FEATURES,0.911324,0.798891,0.816267,0.806340,/kaggle/working/results/phobert_hybrid_models/...
4,DESIGN,0.943679,0.780876,0.760543,0.768347,/kaggle/working/results/phobert_hybrid_models/...
5,GENERAL,0.834032,0.754474,0.777024,0.763752,/kaggle/working/results/phobert_hybrid_models/...
6,PERFORMANCE,0.847214,0.755225,0.767523,0.760506,/kaggle/working/results/phobert_hybrid_models/...
7,SCREEN,0.949670,0.737047,0.742588,0.738839,/kaggle/working/results/phobert_hybrid_models/...
8,SER&ACC,0.897543,0.679005,0.684973,0.672301,/kaggle/working/results/phobert_hybrid_models/...
9,STORAGE,0.993409,0.540910,0.582728,0.559504,/kaggle/working/results/phobert_hybrid_models/...


In [19]:
compare_all_df = baseline_test_eval_df[[
    "aspect",
    "accuracy",
    "macro_f1"
]].rename(columns={
    "accuracy": "accuracy_baseline",
    "macro_f1": "macro_f1_baseline"
}).merge(
    phobert_test_df[[
        "aspect",
        "accuracy",
        "macro_f1"
    ]].rename(columns={
        "accuracy": "accuracy_single_phobert",
        "macro_f1": "macro_f1_single_phobert"
    }),
    on="aspect",
    how="left"
).merge(
    multi_df[[
        "aspect",
        "accuracy",
        "macro_f1"
    ]].rename(columns={
        "accuracy": "accuracy_multitask",
        "macro_f1": "macro_f1_multitask"
    }),
    on="aspect",
    how="left"
).merge(
    hybrid_test_df[[
        "aspect",
        "accuracy",
        "macro_f1"
    ]].rename(columns={
        "accuracy": "accuracy_hybrid",
        "macro_f1": "macro_f1_hybrid"
    }),
    on="aspect",
    how="left"
)

compare_all_df

,aspect,accuracy_baseline,macro_f1_baseline,accuracy_single_phobert,macro_f1_single_phobert,accuracy_multitask,macro_f1_multitask,accuracy_hybrid,macro_f1_hybrid
0,BATTERY,0.889155,0.732998,0.925105,0.837991,0.899940,0.798544,0.925704,0.840429
1,CAMERA,0.905333,0.726731,0.938886,0.809657,0.932295,0.807404,0.946075,0.847405
2,PRICE,0.871780,0.676251,0.920312,0.813819,0.913122,0.811931,0.916117,0.811317
3,GENERAL,0.738167,0.638904,0.823247,0.748876,0.793289,0.725948,0.834032,0.763752
4,FEATURES,0.866986,0.636654,0.890354,0.775595,0.872978,0.730868,0.911324,0.806340
5,PERFORMANCE,0.757340,0.635270,0.835231,0.757597,0.814859,0.727104,0.847214,0.760506
6,SER&ACC,0.880168,0.619108,0.896345,0.646843,0.876573,0.652131,0.897543,0.672301
7,DESIGN,0.886759,0.594819,0.933493,0.756713,0.919113,0.680073,0.943679,0.768347
8,STORAGE,0.992211,0.575759,0.990413,0.563679,0.991612,0.587631,0.993409,0.559504
9,SCREEN,0.934092,0.567491,0.904733,0.614351,0.935890,0.733489,0.949670,0.738839


In [20]:
valid_mask = compare_all_df["macro_f1_hybrid"].notna()

summary_all_df = pd.DataFrame({
    "model": [
        "Baseline",
        "Single-task PhoBERT",
        "Multi-task PhoBERT",
        "Hybrid MTL + Single-task"
    ],
    "average_macro_f1": [
        compare_all_df.loc[valid_mask, "macro_f1_baseline"].mean(),
        compare_all_df.loc[valid_mask, "macro_f1_single_phobert"].mean(),
        compare_all_df.loc[valid_mask, "macro_f1_multitask"].mean(),
        compare_all_df.loc[valid_mask, "macro_f1_hybrid"].mean()
    ],
    "average_accuracy": [
        compare_all_df.loc[valid_mask, "accuracy_baseline"].mean(),
        compare_all_df.loc[valid_mask, "accuracy_single_phobert"].mean(),
        compare_all_df.loc[valid_mask, "accuracy_multitask"].mean(),
        compare_all_df.loc[valid_mask, "accuracy_hybrid"].mean()
    ]
})

summary_all_df

,model,average_macro_f1,average_accuracy
0,Baseline,0.640398,0.872199
1,Single-task PhoBERT,0.732512,0.905812
2,Multi-task PhoBERT,0.725512,0.894967
3,Hybrid MTL + Single-task,0.756874,0.916477


In [ ]:
final_rows = []

for aspect in aspect_list:

    baseline_val_f1 = baseline_val_eval_df.loc[
        baseline_val_eval_df["aspect"] == aspect,
        "macro_f1"
    ].values[0]

    single_val_f1 = phobert_val_df.loc[
        phobert_val_df["aspect"] == aspect,
        "macro_f1"
    ].values[0]

    val_candidates = {
        "Baseline": baseline_val_f1,
        "Single-task PhoBERT": single_val_f1,
    }

    #  chỉ thêm hybrid nếu có
    if aspect in hybrid_val_df["aspect"].values:
        hybrid_val_f1 = hybrid_val_df.loc[
            hybrid_val_df["aspect"] == aspect,
            "macro_f1"
        ].values[0]

        val_candidates["Hybrid MTL + Single-task"] = hybrid_val_f1

    # ===== SELECT =====
    selected_model = max(val_candidates, key=val_candidates.get)
    selected_val_f1 = val_candidates[selected_model]

    # ===== TEST =====
    test_row = compare_all_df[
        compare_all_df["aspect"] == aspect
    ].iloc[0]

    if selected_model == "Baseline":
        test_accuracy = test_row["accuracy_baseline"]
        test_macro_f1 = test_row["macro_f1_baseline"]

    elif selected_model == "Single-task PhoBERT":
        test_accuracy = test_row["accuracy_single_phobert"]
        test_macro_f1 = test_row["macro_f1_single_phobert"]

    else:
        test_accuracy = test_row["accuracy_hybrid"]
        test_macro_f1 = test_row["macro_f1_hybrid"]

    final_rows.append({
        "aspect": aspect,
        "selected_model_by_val": selected_model,
        "val_macro_f1": selected_val_f1,
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1
    })

final_best_model_df = pd.DataFrame(final_rows)

final_best_model_df = final_best_model_df.sort_values(
    "test_macro_f1",
    ascending=False
)

In [ ]:
# SAVE MULTI-TASK / HYBRID RESULTS


multi_df.to_csv(
    f"{RESULT_DIR}/multitask_test_results.csv",
    index=False
)

hybrid_val_df.to_csv(
    f"{RESULT_DIR}/hybrid_validation_results.csv",
    index=False
)

hybrid_test_df.to_csv(
    f"{RESULT_DIR}/hybrid_test_results.csv",
    index=False
)

compare_all_df.to_csv(
    f"{RESULT_DIR}/compare_baseline_single_multitask_hybrid.csv",
    index=False
)

summary_all_df.to_csv(
    f"{RESULT_DIR}/summary_baseline_single_multitask_hybrid.csv",
    index=False
)

final_best_model_df.to_csv(
    f"{RESULT_DIR}/final_best_model_per_aspect.csv",
    index=False
)

print("Saved all multi-task and hybrid results.")

Saved all multi-task and hybrid results.
